# AUDIT-X · exact duplicate exposure of the run splits (CPU, about 5 minutes)

Computes, from the run outputs only (no images, no GPU), the share of known evaluation images with an identical or
near-identical training image, for the grouped, naive and cap-none runs. This replaces the approximate Malimg values in
Table 6.

**Attach the committed outputs of:** `FINAL_B_Malimg`, `FINAL_A2_MaleVis_Amendment1`, `FINAL_C_Dumpware10` and the three
`rq1-naive-vs-grouped-*` notebooks (and `RQ1b_MaleVis_CAP_NONE` if it has run). Accelerator **None**. Commit, then send
`RQ1_exposure_exact.csv` from the Output tab.


In [1]:
%%writefile /kaggle/working/exposure_exact.py
"""Exact duplicate exposure of run splits (no GPU, no images): share of known evaluation images (calibration + test)
that have a training image with an identical file or pixels, an identical 256-bit hash, or a hash within 8 bits
(any family, as in the residual-duplicate table of the paper).

Usage: python exposure_exact.py OUT.csv  DATASET RUN_DIR [RUN_DIR ...]  [-- DATASET RUN_DIR ...]
Each RUN_DIR is a training-run output folder with image_keys.json, MANIFEST.json and split<s>/split.json.
All run folders of one dataset must share the same image keys and unknown families.
"""
import json, os, sys
import numpy as np, pandas as pd

def splits(d):
    return {int(x[5:]): json.load(open(f"{d}/{x}/split.json")) for x in sorted(os.listdir(d))
            if x.startswith("split") and os.path.exists(f"{d}/{x}/split.json")}

_POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def popcount64(x):  # works on NumPy < 2.0, where np.bitwise_count is missing
    if hasattr(np, "bitwise_count"): return np.bitwise_count(x)
    return _POP[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)

def exposure(keys, sp):
    fm = np.array([k[0] for k in keys]); pm = np.array([k[1] for k in keys]); dh = np.array([k[2] for k in keys])
    w = np.frombuffer(bytes.fromhex("".join(dh)), dtype=">u8").reshape(len(dh), 4)
    tr = np.asarray(sp["train"], int); ev = np.concatenate([sp["calib"], sp["test"]]).astype(int)
    exact = np.isin(fm[ev], fm[tr]) | np.isin(pm[ev], pm[tr]); same = np.isin(dh[ev], dh[tr]); near = np.zeros(len(ev), bool)
    for i in range(0, len(ev), 300):
        near[i:i + 300] = (popcount64(w[ev[i:i + 300]][:, None, :] ^ w[tr][None, :, :]).sum(2) <= 8).any(1)
    return 100 * exact.mean(), 100 * same.mean(), 100 * near.mean()

rows = []; groups = []; cur = []
for a in sys.argv[2:]:
    if a == "--": groups.append(cur); cur = []
    else: cur.append(a)
groups.append(cur)
for g in groups:
    ds, dirs = g[0], g[1:]
    keys0 = json.load(open(f"{dirs[0]}/image_keys.json"))["keys"]; S0 = splits(dirs[0])
    for d in dirs:
        man = json.load(open(f"{d}/MANIFEST.json")); keys = json.load(open(f"{d}/image_keys.json"))["keys"]; S = splits(d)
        assert keys == keys0, f"{ds}: {d} has different image keys"
        for s in S:
            assert S[s]["unknown_names"] == S0[s]["unknown_names"] and sorted(S[s]["unk"]) == sorted(S0[s]["unk"]), f"{ds} split {s}: unknown families differ"
            e = exposure(keys, S[s])
            rows.append(dict(dataset=ds, protocol=man["protocol"], split=s, exact=e[0], identical_hash=e[1], within8=e[2]))
        print(f"{ds}: {man['protocol']} done", flush=True)
D = pd.DataFrame(rows); D.to_csv(sys.argv[1], index=False)
print(D.groupby(["dataset", "protocol"])[["exact", "identical_hash", "within8"]].mean().round(2).to_string())


Writing /kaggle/working/exposure_exact.py


In [2]:
import glob, os, subprocess, sys
NAMES = {"Malimg": ["auditx_final_malimg", "auditx_naive_malimg"],
         "MaleVis": ["auditx_final_malevis_v5", "auditx_naive_malevis", "auditx_capnone_malevis"],
         "Dumpware10": ["auditx_final_dumpware10", "auditx_naive_dumpware10"]}
args = []
for ds, names in NAMES.items():
    dirs = []
    for n in names:
        hits = [h for h in glob.glob(f"/kaggle/input/**/{n}", recursive=True) if os.path.exists(os.path.join(h, "MANIFEST.json"))]
        print(ds, n, "->", hits[0] if hits else "NOT ATTACHED")
        if hits: dirs.append(hits[0])
    if len(dirs) >= 2: args += (["--"] if args else []) + [ds] + dirs
assert args, "attach the grouped and naive run outputs"
r = subprocess.run([sys.executable, "/kaggle/working/exposure_exact.py", "/kaggle/working/RQ1_exposure_exact.csv"] + args, capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-2000:])


Malimg auditx_final_malimg -> /kaggle/input/notebooks/mohammadmehedimunna/final-b-malimg/auditx_final_malimg
Malimg auditx_naive_malimg -> /kaggle/input/notebooks/rauf41/rq1-naive-vs-grouped-malimg/auditx_naive_malimg
MaleVis auditx_final_malevis_v5 -> /kaggle/input/notebooks/rauf41/final-a2-malevis-amendment1/auditx_final_malevis_v5
MaleVis auditx_naive_malevis -> /kaggle/input/notebooks/rauf41/rq1-naive-vs-grouped-malevis/auditx_naive_malevis
MaleVis auditx_capnone_malevis -> /kaggle/input/notebooks/rauf41/rq1b-malevis-cap-none/auditx_capnone_malevis
Dumpware10 auditx_final_dumpware10 -> /kaggle/input/notebooks/rauf41/final-c-dumpware10/auditx_final_dumpware10
Dumpware10 auditx_naive_dumpware10 -> /kaggle/input/notebooks/rauf41/rq1-naive-vs-grouped-dumpware10/auditx_naive_dumpware10
Malimg: grouped-L1-v3+near-L3-8bit-cap250 done
Malimg: grouped-L1-v3+near-L3-8bit-cap250+NAIVE-image-level-split done
MaleVis: grouped-L1-v3+identical-hash-whole+near-L3-8bit-cap250 done
MaleVis: grouped-